In [ ]:
Oracle AI Data Platform v1.0

Copyright © 2025, Oracle and/or its affiliates.

Licensed under the Universal Permissive License v 1.0 as shown at https://oss.oracle.com/licenses/upl/

# HubSpot Connector Samples

Read-only samples that ingest HubSpot CRM contacts, companies, deals and their associations into Delta tables with Python and the HubSpot REST API, as there is no built-in HubSpot connector type. Replace all placeholders before running.

## Prerequisites

1. Create a HubSpot service key, or a legacy private app token, with the scopes `crm.objects.contacts.read`, `crm.objects.companies.read` and `crm.objects.deals.read`.
2. Create a **Secret Token** credential in the Credential Store with key `<KEY>` holding that token.
3. The cluster needs outbound HTTPS access to `api.hubapi.com`. The `requests` package is already on the cluster, so no library install is needed.
4. The target catalog must already exist. The schema is created if missing.
5. Run one instance at a time. Two overlapping runs would write the same tables and watermarks.

## Configuration

Set the variables below. `MODE` is `incremental` or `full`. An object with no stored watermark is loaded in full on its first incremental run. Associations are always re-read in full and overwrite their table, so run them after the objects they start from.

In [ ]:
import json, math, re, time, uuid
from itertools import chain
from datetime import datetime, timedelta, timezone
from decimal import Decimal, InvalidOperation
import requests

CREDENTIAL_NAME = "<CREDENTIAL_NAME>"
CREDENTIAL_KEY = "<KEY>"
CATALOG = "<CATALOG>"
SCHEMA = "<SCHEMA>"
TABLE_PREFIX = ""                  # optional prefix for every table name
OBJECTS = ["contacts", "companies", "deals", "associations"]
MODE = "incremental"               # "incremental" or "full"
OVERLAP_SECONDS = 300              # incremental re-reads this far before the watermark
REQUESTS_PER_SECOND = 4.0          # client-side throttle, HubSpot search allows 5 per second

API_URL = "https://api.hubapi.com"
API_VERSION = "2026-03"
MODIFIED = {"contacts": "lastmodifieddate", "companies": "hs_lastmodifieddate", "deals": "hs_lastmodifieddate"}
ASSOCIATION_PAIRS = [("contacts", "companies"), ("deals", "contacts"), ("deals", "companies")]
SEARCH_CAP, SEARCH_PAGE, READ_BATCH, ASSOC_BATCH, URL_LIMIT = 10000, 200, 100, 1000, 15000

for _name, _value, _pattern in (("CATALOG", CATALOG, r"\w+"), ("SCHEMA", SCHEMA, r"\w+"), ("TABLE_PREFIX", TABLE_PREFIX, r"\w*")):
    if not re.fullmatch(_pattern, _value):
        raise ValueError(f"{_name} must contain only letters, digits and underscores, got {_value!r}. Replace any placeholder.")
if MODE not in ("incremental", "full"):
    raise ValueError(f"MODE must be 'incremental' or 'full', got {MODE!r}")
if not set(OBJECTS) <= {"contacts", "companies", "deals", "associations"}:
    raise ValueError(f"OBJECTS has unknown entries: {sorted(set(OBJECTS) - {'contacts', 'companies', 'deals', 'associations'})}")

try:
    if isinstance(OVERLAP_SECONDS, bool) or not isinstance(OVERLAP_SECONDS, (int, float)) or OVERLAP_SECONDS < 0:
        raise ValueError
    timedelta(seconds=OVERLAP_SECONDS)  # raises for NaN, inf and values too large
except (OverflowError, ValueError):
    raise ValueError(f"OVERLAP_SECONDS must be 0 or more, got {OVERLAP_SECONDS!r}") from None

token = aidputils.secrets.get(name=CREDENTIAL_NAME, key=CREDENTIAL_KEY)
http = requests.Session()
http.headers.update({"Authorization": f"Bearer {token}", "Content-Type": "application/json"})

## HubSpot API Client

`hs_call` sends one throttled request with retries, and `hs_pages` follows the cursor of a list endpoint.

In [ ]:
_next_call = [0.0]

def _backoff(n):
    return min(2 ** (n - 1), 30)

def hs_call(method, path, body=None, params=None):
    failures = waits = 0
    while True:
        gap = _next_call[0] - time.monotonic()
        if gap > 0:
            time.sleep(gap)
        _next_call[0] = time.monotonic() + 1.0 / REQUESTS_PER_SECOND
        try:
            resp = http.request(method, API_URL + path, json=body, params=params, timeout=60)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout):
            failures += 1
            if failures >= 8:
                raise
            time.sleep(_backoff(failures))
            continue
        status = resp.status_code
        if status < 400:
            return resp.json()  # includes 207: the caller reads "errors"
        try:
            err = resp.json()
            category, message, policy = str(err.get("category")), str(err.get("message")), err.get("policyName")
        except ValueError:
            category, message, policy = "unknown", "response body was not JSON", None
        if status == 429 and policy != "DAILY" and waits < 20:
            waits += 1
            try:  # HubSpot usually sends no Retry-After; honor it when it is a number of seconds
                delay = float(resp.headers.get("Retry-After"))
                if not math.isfinite(delay) or delay < 0:
                    raise ValueError
                delay = min(delay, 30)
            except (TypeError, ValueError):
                delay = _backoff(waits)
            time.sleep(delay)
        elif status in (500, 502, 503, 504) and failures < 7:
            failures += 1
            time.sleep(_backoff(failures))
        else:
            raise RuntimeError(f"HubSpot API error {status} ({category}): {message}")

def hs_pages(path, params):
    cursor, seen = None, set()
    while True:
        page = hs_call("GET", path, params=dict(params, limit=100, **({"after": cursor} if cursor else {})))
        yield from page.get("results") or []
        cursor = ((page.get("paging") or {}).get("next") or {}).get("after")
        if not cursor or cursor in seen:  # a missing or repeated cursor ends the loop
            return
        seen.add(cursor)

## Extraction Functions

These functions find changed ids with search, fetch full records with batch read, list archived records and read associations.

In [ ]:
def objects_path(obj):
    return f"/crm/objects/{API_VERSION}/{obj}"

def check_207(payload, tolerated):
    for e in payload.get("errors") or []:
        if str(e.get("category")) not in tolerated:
            raise RuntimeError(f"HubSpot API error 207 ({e.get('category')}): {e.get('message')}")

def batch_read(obj, properties, ids, archived=False):
    body = {"properties": properties, "inputs": [{"id": i} for i in ids]}
    payload = hs_call("POST", objects_path(obj) + "/batch/read", body=body, params={"archived": "true"} if archived else None)
    check_207(payload, ("OBJECT_NOT_FOUND",))
    return payload.get("results") or []

def read_by_ids(obj, properties, ids, archived=False):
    batch = []
    for i in ids:
        batch.append(str(i))
        if len(batch) == READ_BATCH:
            yield from batch_read(obj, properties, batch, archived)
            batch = []
    if batch:
        yield from batch_read(obj, properties, batch, archived)

def iter_all(obj, properties, archived=False):
    flag = "true" if archived else "false"
    joined = ",".join(properties)
    if len(joined) <= URL_LIMIT:
        return hs_pages(objects_path(obj), {"properties": joined, "archived": flag})
    # too many properties for a URL: list ids only, then fetch the records by batch
    ids = (r["id"] for r in hs_pages(objects_path(obj), {"archived": flag}))
    return read_by_ids(obj, properties, ids, archived)

def iter_archived(obj):
    for r in hs_pages(objects_path(obj), {"archived": "true"}):
        yield str(r["id"]), to_time(r.get("archivedAt"))

def search_ids(obj, since, until):
    ms = lambda d: str(int(d.timestamp() * 1000))
    prop = MODIFIED[obj]
    window = [{"propertyName": prop, "operator": "GTE", "value": ms(since)},
              {"propertyName": prop, "operator": "LT", "value": ms(until)}]
    last_id = None
    while True:
        filters = window + ([{"propertyName": "hs_object_id", "operator": "GT", "value": str(last_id)}] if last_id else [])
        body = {"filterGroups": [{"filters": filters}], "properties": ["hs_object_id"], "limit": SEARCH_PAGE,
                "sorts": [{"propertyName": "hs_object_id", "direction": "ASCENDING"}]}
        seen, cursor = 0, None
        while True:
            page = hs_call("POST", objects_path(obj) + "/search", body=dict(body, after=cursor) if cursor else body)
            for r in page.get("results") or []:
                last_id = int(r["id"])
                seen += 1
                yield str(r["id"])
            cursor = ((page.get("paging") or {}).get("next") or {}).get("after")
            if not cursor or seen >= SEARCH_CAP:
                break
        if seen < SEARCH_CAP:
            return

def iter_associations(from_obj, to_obj, from_ids):
    path = f"/crm/associations/{API_VERSION}/{from_obj}/{to_obj}/batch/read"
    pending = [{"id": str(i)} for i in from_ids]
    while pending:
        following = []
        for start in range(0, len(pending), ASSOC_BATCH):
            payload = hs_call("POST", path, body={"inputs": pending[start:start + ASSOC_BATCH]})
            check_207(payload, ("NO_ASSOCIATIONS_FOUND", "OBJECT_NOT_FOUND"))
            for res in payload.get("results") or []:
                from_id = str(res["from"]["id"])
                for target in res.get("to") or []:
                    for kind in target.get("associationTypes") or []:
                        yield from_id, str(target["toObjectId"]), kind
                cursor = ((res.get("paging") or {}).get("next") or {}).get("after")
                if cursor:
                    following.append({"id": from_id, "after": cursor})
        pending = following

## Row Mapping and Delta Writer

These functions map HubSpot records to typed rows, then stage, merge or overwrite them in Delta and flag archived ids.

In [ ]:
COMMON = [("id", "STRING"), ("created_at", "TIMESTAMP"), ("updated_at", "TIMESTAMP"),
          ("archived", "BOOLEAN"), ("archived_at", "TIMESTAMP")]
TAIL = [("raw_json", "STRING"), ("_ingested_at", "TIMESTAMP")]
COLUMNS = {
    "contacts": COMMON + [("email", "STRING"), ("first_name", "STRING"), ("last_name", "STRING"),
                          ("phone", "STRING"), ("lifecycle_stage", "STRING")] + TAIL,
    "companies": COMMON + [("name", "STRING"), ("domain", "STRING"), ("industry", "STRING")] + TAIL,
    "deals": COMMON + [("deal_name", "STRING"), ("amount", "DECIMAL(38,6)"), ("currency", "STRING"),
                       ("deal_stage", "STRING"), ("pipeline", "STRING"), ("close_date", "TIMESTAMP"),
                       ("owner_id", "STRING")] + TAIL,
    "associations": [("from_object", "STRING"), ("from_id", "STRING"), ("to_object", "STRING"), ("to_id", "STRING"),
                     ("association_type_id", "INT"), ("category", "STRING"), ("label", "STRING"),
                     ("_ingested_at", "TIMESTAMP")],
}
ASSOCIATION_KEY = ["from_object", "from_id", "to_object", "to_id", "association_type_id"]
PROPERTY_MAP = {  # column -> HubSpot property
    "contacts": {"email": "email", "first_name": "firstname", "last_name": "lastname", "phone": "phone",
                 "lifecycle_stage": "lifecyclestage"},
    "companies": {"name": "name", "domain": "domain", "industry": "industry"},
    "deals": {"deal_name": "dealname", "currency": "deal_currency_code", "deal_stage": "dealstage",
              "pipeline": "pipeline", "owner_id": "hubspot_owner_id"},
}
EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)

def to_time(value):
    """ISO 8601 string or epoch milliseconds to an aware UTC datetime, else None."""
    try:
        if value is None or isinstance(value, bool):
            return None
        text = str(value).strip()
        if not text:
            return None
        if re.fullmatch(r"-?\d+", text):
            return EPOCH + timedelta(milliseconds=int(text))
        text = re.sub(r"[Zz]$", "+00:00", text)
        text = re.sub(r"\.(\d+)", lambda m: "." + m.group(1)[:6].ljust(6, "0"), text, count=1)
        parsed = datetime.fromisoformat(text)
        return (parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)).astimezone(timezone.utc)
    except (ValueError, OverflowError, OSError):
        return None

def to_amount(value):
    """Round to 6 places; None if unparseable or too large for DECIMAL(38,6)."""
    try:
        number = Decimal(str(value).strip()).quantize(Decimal("0.000001"))
    except (InvalidOperation, ValueError):
        return None
    return number if number.is_finite() and number.adjusted() <= 31 else None

def to_text(value):
    return None if value is None else str(value)

def object_row(obj, rec):
    props = rec.get("properties") if isinstance(rec.get("properties"), dict) else {}
    archived = rec.get("archived") is True
    row = {"id": to_text(rec.get("id")), "created_at": to_time(rec.get("createdAt")),
           "updated_at": to_time(rec.get("updatedAt")), "archived": archived,
           "archived_at": to_time(rec.get("archivedAt")) if archived else None,
           "raw_json": json.dumps(rec, sort_keys=True, ensure_ascii=False, separators=(",", ":")),
           "_ingested_at": run_at}
    row.update({col: to_text(props.get(prop)) for col, prop in PROPERTY_MAP[obj].items()})
    if obj == "deals":
        row["amount"], row["close_date"] = to_amount(props.get("amount")), to_time(props.get("closedate"))
    return row

def association_row(from_obj, from_id, to_obj, to_id, kind):
    try:
        type_id = int(kind.get("typeId"))
    except (TypeError, ValueError):
        type_id = None
    return {"from_object": from_obj, "from_id": from_id, "to_object": to_obj, "to_id": to_id,
            "association_type_id": type_id, "category": to_text(kind.get("category")),
            "label": to_text(kind.get("label")), "_ingested_at": run_at}

def merged_ids(rec):
    raw = (rec.get("properties") or {}).get("hs_merged_object_ids")
    return list(dict.fromkeys(p.strip() for p in str(raw or "").split(";") if p.strip()))

def table_name(name):
    return f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}{name}"

def ddl(columns):
    return ", ".join(f"{n} {t}" for n, t in columns)

def stage(staging, rows, columns, batch_size=5000):
    spark.sql(f"DROP TABLE IF EXISTS {staging}")
    spark.sql(f"CREATE TABLE {staging} ({ddl(columns)}) USING DELTA")
    names, total, batch = [n for n, _ in columns], 0, []
    for row in rows:
        batch.append(tuple(row[n] for n in names))
        if len(batch) >= batch_size:
            spark.createDataFrame(batch, ddl(columns)).write.format("delta").mode("append").saveAsTable(staging)
            total, batch = total + len(batch), []
    if batch:
        spark.createDataFrame(batch, ddl(columns)).write.format("delta").mode("append").saveAsTable(staging)
        total += len(batch)
    return total

def write_table(name, rows, key=("id",), order_by="updated_at", overwrite=False):
    table, staging = table_name(name), f"{table_name(name)}__staging_{run_id}"
    columns = COLUMNS[name]
    spark.sql(f"CREATE TABLE IF NOT EXISTS {table} ({ddl(columns)}) USING DELTA")
    try:
        count = stage(staging, rows, columns)
        # Delta rejects a MERGE whose source holds two rows for one key, so keep the newest
        source = (f"SELECT {', '.join(n for n, _ in columns)} FROM (SELECT *, ROW_NUMBER() OVER "
                  f"(PARTITION BY {', '.join(key)} ORDER BY {order_by} DESC) AS _rn FROM {staging}) WHERE _rn = 1")
        if overwrite:
            spark.sql(f"INSERT OVERWRITE TABLE {table} {source}")
        else:
            on = " AND ".join(f"t.{k} = s.{k}" for k in key)
            spark.sql(f"MERGE INTO {table} t USING ({source}) s ON {on} "
                      "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {staging}")
    return count

def mark_archived(name, pairs):
    table, staging = table_name(name), f"{table_name(name)}__archived_{run_id}"
    columns = [("id", "STRING"), ("archived_at", "TIMESTAMP")]
    try:
        if stage(staging, ({"id": i, "archived_at": at} for i, at in pairs), columns) == 0:
            return
        source = ("SELECT id, archived_at FROM (SELECT *, ROW_NUMBER() OVER (PARTITION BY id "
                  f"ORDER BY archived_at DESC NULLS LAST) AS _rn FROM {staging}) WHERE _rn = 1")
        spark.sql(f"MERGE INTO {table} t USING ({source}) s ON t.id = s.id "
                  "WHEN MATCHED THEN UPDATE SET archived = true, archived_at = s.archived_at")
    finally:
        spark.sql(f"DROP TABLE IF EXISTS {staging}")

## Run the Ingestion

This cell syncs each selected object, keeps a watermark per object in `hubspot_sync_state`, and raises at the end if any object failed.

In [ ]:
STATE = table_name("hubspot_sync_state")
STATE_DDL = "object_name STRING, watermark TIMESTAMP, last_mode STRING, last_status STRING, last_rows BIGINT, last_run_at TIMESTAMP"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE TABLE IF NOT EXISTS {STATE} ({STATE_DDL}) USING DELTA")

def get_watermark(name):
    # read microseconds so the result does not depend on the Spark session time zone
    rows = spark.sql(f"SELECT unix_micros(watermark) FROM {STATE} WHERE object_name = '{name}'").collect()
    return EPOCH + timedelta(microseconds=int(rows[0][0])) if rows and rows[0][0] is not None else None

def set_watermark(name, mode, count):
    spark.createDataFrame([(name, run_at, mode, "SUCCESS", count, run_at)], STATE_DDL).createOrReplaceTempView("_hubspot_state_update")
    spark.sql(f"MERGE INTO {STATE} t USING _hubspot_state_update s ON t.object_name = s.object_name "
              "WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")

def set_failed(name, mode):
    # records the failure but leaves watermark and last_rows as they were
    spark.createDataFrame([(name, None, mode, "FAILED", None, run_at)], STATE_DDL).createOrReplaceTempView("_hubspot_state_failed")
    spark.sql(f"MERGE INTO {STATE} t USING _hubspot_state_failed s ON t.object_name = s.object_name "
              "WHEN MATCHED THEN UPDATE SET last_mode = s.last_mode, last_status = s.last_status, last_run_at = s.last_run_at "
              "WHEN NOT MATCHED THEN INSERT *")

def sync_object(name, mode, since):
    properties = [p["name"] for p in hs_call("GET", f"/crm/properties/{API_VERSION}/{name}").get("results") or [] if p.get("name")]
    if mode == "full":
        records = chain(iter_all(name, properties), iter_all(name, properties, archived=True))
        return write_table(name, (object_row(name, r) for r in records), overwrite=True)
    merged = []
    def rows():
        for r in read_by_ids(name, properties, search_ids(name, since - timedelta(seconds=OVERLAP_SECONDS), run_at)):
            merged.extend(merged_ids(r))
            yield object_row(name, r)
    count = write_table(name, rows())
    # after the merge, so the table exists; merged-away ids vanish from HubSpot without being archived
    mark_archived(name, list(iter_archived(name)) + [(i, run_at) for i in merged])
    return count

def sync_associations():
    sources, missing = {}, []
    for pair in ASSOCIATION_PAIRS:
        if pair[0] not in OBJECTS:
            print(f"associations: skipping {pair[0]}->{pair[1]}, {pair[0]} is not in OBJECTS")
            continue
        try:
            sources[pair] = [str(r[0]) for r in spark.sql(f"SELECT id FROM {table_name(pair[0])} WHERE NOT archived").collect()]
        except Exception:
            missing.append(f"{pair[0]}->{pair[1]}: source table {table_name(pair[0])} cannot be read, sync it first")
    def rows():
        for (from_obj, to_obj), ids in sources.items():
            for from_id, to_id, kind in iter_associations(from_obj, to_obj, ids):
                yield association_row(from_obj, from_id, to_obj, to_id, kind)
    # a full overwrite would drop the old links of a skipped or failed pair, so a partial run only merges
    count = write_table("associations", rows(), key=ASSOCIATION_KEY, order_by="_ingested_at", overwrite=len(sources) == len(ASSOCIATION_PAIRS))
    if missing:
        raise RuntimeError("; ".join(missing) + f" ({count} row(s) merged for the other pairs)")
    return count

run_at = datetime.now(timezone.utc)  # the next watermark: the run start, not the newest record seen
run_id = uuid.uuid4().hex[:8]
failed = []
for name in [o for o in ("contacts", "companies", "deals", "associations") if o in OBJECTS]:
    mode = "full"
    try:
        since = get_watermark(name) if name != "associations" else None
        mode = "full" if name == "associations" or MODE == "full" or since is None else "incremental"
        count = sync_associations() if name == "associations" else sync_object(name, mode, since)
        set_watermark(name, mode, count)
    except Exception as exc:  # one object failing must not stop the others
        failed.append(f"{name}: {str(exc)[:1000]}")
        print(f"{name}: FAILED, {failed[-1]}")
        try:
            set_failed(name, mode)
        except Exception as state_exc:  # best effort: the failure is already recorded in `failed`
            print(f"{name}: could not record FAILED state, {str(state_exc)[:200]}")
        continue
    print(f"{name}: {count} row(s), {mode}, watermark {run_at.isoformat()}")

spark.sql(f"SELECT * FROM {STATE}").show(truncate=False)
if failed:
    raise RuntimeError("HubSpot sync failed for " + "; ".join(failed))

## Query the Tables

Read a synced table back with Spark.

In [ ]:
contacts_df = spark.table(table_name("contacts"))
contacts_df.show()

## Connector Options

The notebook is Python, so the options are its configuration variables.

| Parameter name | Valid values | Mandatory | Description |
| --- | --- | --- | --- |
| `CREDENTIAL_NAME` | Credential Store entry name | Y | Credential holding the HubSpot token. |
| `CREDENTIAL_KEY` | Key inside that credential | Y | Key that holds the token. |
| `CATALOG` | Existing catalog name | Y | Target catalog. It must already exist. |
| `SCHEMA` | Schema name | Y | Target schema. Created if missing. |
| `TABLE_PREFIX` | Letters, digits and underscores | N | Prefix for every table name. Default is empty. |
| `OBJECTS` | List of `contacts`, `companies`, `deals`, `associations` | Y | Objects to ingest. |
| `MODE` | `incremental`, `full` | Y | Incremental merges changes since the watermark. Full overwrites the tables. |
| `OVERLAP_SECONDS` | Integer of 0 or more | N | How far before the watermark an incremental run re-reads. Default is 300. |
| `REQUESTS_PER_SECOND` | Number greater than 0 | N | Client-side throttle. Default is 4. |
| `API_VERSION` | HubSpot dated API version | N | Dated CRM API version used in every path. Default is 2026-03. |